# UQ Failure Analysis Large Language Model
---

## Imports

In [1]:
import json

import numpy as np
import torch

from reionemu import (
    load_training_arrays,
    MCDropoutEmulator,
    predict_mc,
)

## Paths and Constants

In [2]:
h5_path = "../data/condensed_v6.h5"
llm_path = "../results/llm/"

# Base Model
ckpt_path = "../checkpoints/base_model/checkpoint.pt"
norm_path = "../checkpoints/base_model/norm/"
split_idx_path = "../checkpoints/base_model/split_idx/split_idx.npz"

In [3]:
SEED = 42

modelcfg = {
    "input_dim": 4,
    "output_dim": 5,
    "hidden_dim": 20,
    "num_hidden_layers": 2,
    "activation": "relu",
    "dropout_rate": 0.1,
}

## Load Data

In [4]:
X, Y, ell = load_training_arrays(h5_path)

X_mean = np.load(norm_path + "X_mean.npy")
X_std = np.load(norm_path + "X_std.npy")

splits_idx = np.load(split_idx_path)
train_idx = splits_idx["train_idx"]
test_idx = splits_idx["test_idx"]

X_train, Y_train = X[train_idx], Y[train_idx]
X_test, Y_test = X[test_idx], Y[test_idx]

## Load Model

In [5]:
state_dict = torch.load(ckpt_path, weights_only=False)

model = MCDropoutEmulator(**modelcfg)
model.load_state_dict(state_dict)

model.eval()

MCDropoutEmulator(
  (network): Sequential(
    (0): Linear(in_features=4, out_features=20, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=20, out_features=20, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.1, inplace=False)
    (6): Linear(in_features=20, out_features=5, bias=True)
  )
)

## Make Predictions

In [6]:
np.random.seed(SEED)
torch.manual_seed(SEED)

pred_mean, pred_std, _, _ = predict_mc(X_test, model, X_mean, X_std, n_mc_samples=201, device="cpu")

## Select and Save Simulations

In [7]:
rng = np.random.default_rng(SEED)
others = rng.choice(np.setdiff1d(np.arange(len(test_idx)), [149, 50]), size=23, replace=False)
llm_test_pos = rng.permutation(np.concatenate([[149, 50], others]))
np.save(llm_path + "test_positions.npy", llm_test_pos)

ell_cols = ",".join(f"D_ell_{int(l)}" for l in ell)
train_rows = np.column_stack([X_train, np.exp(Y_train)])
np.savetxt(llm_path + "train.txt", train_rows, delimiter=",", header="zmean,alpha,kb,b0," + ell_cols, comments="", fmt="%.4g")

test_rows = np.column_stack([np.arange(len(llm_test_pos)), X_test[llm_test_pos]])
np.savetxt(llm_path + "test_inputs.txt", test_rows, delimiter=",", header="id,zmean,alpha,kb,b0", comments="", fmt=["%d", "%.4g", "%.4g", "%.4g", "%.4g"])

# Prompt for Frontier Model
---
```text
Attached are two files.

train.txt contains 700 reionization simulations. Each row has four input parameters (zmean, alpha, kb, b0) followed by the resulting kSZ angular power spectrum D_ell, in units of micro kelvin, at five ell bin centers: ell = 2033.50, ell = 4093.42, ell = 6153.33, ell = 8213.25, and ell = 10273.17.

test_inputs.txt contains 25 new parameter sets, with ids 0 to 24. For each one, predict D_ell at all five ell values and give a 1-sigma uncertainty for each prediction. A 1-sigma uncertainty means you expect the true value to fall within +/-1 sigma about 68% of the time.

Base your predictions only on the training data. Do not write or run code.

Return only a JSON, with no other text. It must contain exactly 25 objects, one per test id, in order from id 0 to id 24. Each object has three fields:

  "id":   the test id (integer)
  "mean": a list of 5 numbers, the predicted D_ell at ell = 2033, 4093, 6153, 8213, 10273, in that order
  "std":  a list of 5 positive numbers, the 1-sigma uncertainty for each of those predictions, in the same order

Example of the structure (placeholder values):
[
  {"id": 0, "mean": [m1, m2, m3, m4, m5], "std": [s1, s2, s3, s4, s5]},
  {"id": 1, "mean": [m1, m2, m3, m4, m5], "std": [s1, s2, s3, s4, s5]},
  ...
  {"id": 24, "mean": [m1, m2, m3, m4, m5], "std": [s1, s2, s3, s4, s5]}
]
```
## Screenshot of ChatGPT-6
![ChatGPT-Chat](../results/figures/llm/chatgpt-chat.png)

## Read LLM Output

In [8]:
with open(llm_path + "llm_predictions.json") as f:
    llm = json.load(f)

llm_mean = np.array([d["mean"] for d in llm])
llm_std = np.array([d["std"] for d in llm])

In [9]:
true = np.exp(Y_test[llm_test_pos])
mc_mean = pred_mean[llm_test_pos]
mc_std = pred_std[llm_test_pos]

mc_rel_error = np.abs(mc_mean - true) / true
llm_rel_error = np.abs(llm_mean - true) / true

mc_abs_pulls = np.abs(mc_mean - true) / mc_std
llm_abs_pulls = np.abs(llm_mean - true) / llm_std

## Mean Relative Error

In [10]:
print(f"Mean Relative Error:\tMC-dropout: {mc_rel_error.mean():.3f}\tLLM: {llm_rel_error.mean():.3f}")

Mean Relative Error:	MC-dropout: 0.059	LLM: 0.062


## Coverage

In [11]:
for k in (1, 2, 3):
    print(f"{k} sigma:\tMC-dropout {np.mean(mc_abs_pulls < k):.2f}\tLLM {np.mean(llm_abs_pulls < k):.2f}")

1 sigma:	MC-dropout 0.58	LLM 0.76
2 sigma:	MC-dropout 0.85	LLM 0.95
3 sigma:	MC-dropout 0.92	LLM 0.99
